# optimiser Module

This module contains the core functionality for the Gaussian Process. The various kernels are initialised and evaluated against each other.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import os
from contextlib import redirect_stderr


from typing import Dict, Tuple
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, ConstantKernel, Kernel, Matern, RationalQuadratic, 
    WhiteKernel, DotProduct
)
from sklearn.model_selection import cross_val_score
from scipy.optimize import minimize
from scipy.stats import qmc
from .base import BaseBayesianOptimizer, Prediction

## lbfgs_optimizer

This function creates a LatinHypercube of x values and runs the objective function for each of them. The top n_restarts candidates are retained and the L-BFGS-B method is used to refine each to its local optimum. The function returns the set of x values that minimises the objective function.

In [ ]:
def lbfgs_optimizer(obj_func, bounds_val, n_dims, n_restarts=20) -> Tuple[np.ndarray, float]:
 
    bounds = [bounds_val] * n_dims
    
    sampler = qmc.LatinHypercube(d=n_dims, seed=42)
    candidates = qmc.scale(sampler.random(n=1000), [b[0] for b in bounds], [b[1] for b in bounds])
    
    best_x = None
    best_val = float('inf')

    vals = []
    for c in candidates:
        try:
            vals.append(obj_func(c))
        except:
            vals.append(1e10)

    top_indices = np.argsort(vals)[:n_restarts]

    for idx in top_indices:
        x0 = candidates[idx]
        try:
            res = minimize(obj_func, x0, bounds=bounds, method='L-BFGS-B')
            if res.fun < best_val:
                best_val = res.fun
                best_x = res.x
        except:
            continue
            
    return best_x, best_val

## BayesianOptimizer

The BayesianOptimizer is defined as a subclass of the BaseBayesianOptimizer in the base module.

### global_optimiser_wrapper

This is a custom optimiser wrapper used by the GaussianProcessorRegressor class to call the custom optimiser defined above. The obj_func argument is log_marginal_likelihood by default. initial_theta/bounds are required arguments from the GaussianProcessRegressor class but are not used here.

### _get_kernels

This defines the kernels to be evaluated: a simple Matérn kernel, RationalQuadratic, and two combined kernels, Matérn + Linear and Matérn + RBF. Bounds are defined to constrain the parameters the optimiser can output for each kernel.

### evaluate_models

This initialises the GaussianProcessorRegressor class for each of the kernels and evaluates each by cross-validation on the negative mean squared error and the log marginal likelihood. It returns a list of models sorted on LML.

### predict_normalized

A function used by the plotter class to return the mean and standard deviation for a given set of x values.

In [ ]:
class BayesianOptimizer(BaseBayesianOptimizer):
    
    @staticmethod 
    def global_optimiser_wrapper(obj_func, initial_theta, bounds):
        x, _ = lbfgs_optimizer(
            lambda t: obj_func(t, eval_gradient=False), 
            bounds[0],
            len(bounds)
        )
        return x, obj_func(x, eval_gradient=False)
        
    def _get_kernels(self) -> Dict[str, Kernel]:
        common_bound = (1e-4, 1e5) 
        noise = WhiteKernel(noise_level=1e-2, noise_level_bounds=(1e-6, 1.0))
        amp = ConstantKernel(constant_value=1.0, constant_value_bounds=common_bound)
        ard_dims = np.full(self.n_dims, 1.0) 
        kernels = {}

        kernels["Matern 2.5"] = amp * Matern(
            length_scale=ard_dims, length_scale_bounds=common_bound, nu=2.5
        ) + noise

        kernels["RationalQuadratic"] = amp * RationalQuadratic(
            length_scale=1.0, 
            alpha=1.0, 
            length_scale_bounds=common_bound,
            alpha_bounds=common_bound 
        ) + noise

        kernels["Linear + Matern"] = (
            ConstantKernel(1.0, common_bound) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-5, 1e5)) + 
            amp * Matern(length_scale=ard_dims, length_scale_bounds=common_bound, nu=2.5)
        ) + noise

        kernels["Mixture"] = (
            (ConstantKernel(1.0, common_bound) * RBF(length_scale=ard_dims, length_scale_bounds=common_bound)) +
            (ConstantKernel(1.0, common_bound) * Matern(length_scale=ard_dims, length_scale_bounds=common_bound, nu=1.5))
        ) + noise

        return kernels

    def evaluate_models(self, cv: int = 5) -> Tuple[Dict[str, GaussianProcessRegressor], pd.DataFrame]:
        kernels = self._get_kernels()
        results = []
        models = {}

        for name, kernel in kernels.items():
            gpr = GaussianProcessRegressor(
                kernel=kernel, 
                normalize_y=True,
                optimizer=self.global_optimiser_wrapper, 
                n_restarts_optimizer=0, 
                random_state=self.seed,
                alpha=1e-8
            )
            
            eff_cv = min(cv, len(self.X)) if len(self.X) > 3 else 2
            
            try:
                scores = cross_val_score(gpr, self.X, self.y_norm, cv=eff_cv, scoring="neg_mean_squared_error", n_jobs=-1)
                mean_score = scores.mean()
                std_score = scores.std()
            except:
                mean_score = -np.inf
                std_score = 0.0
            
            try:
                with open(os.devnull, 'w') as f, redirect_stderr(f):
                    gpr.fit(self.X, self.y_norm)
                    lml = gpr.log_marginal_likelihood()
            except:
                lml = -np.inf

            models[name] = gpr
            results.append({
                "name": name,
                "mean_score": mean_score,
                "std_score": std_score,
                "log_marginal_likelihood": lml,
                "final_kernel": str(gpr.kernel_)
            })

        df_results = pd.DataFrame(results).sort_values("log_marginal_likelihood", ascending=False).reset_index(drop=True)
        return models, df_results
    
    def _predict_normalized(self, model: GaussianProcessRegressor, X_candidates: np.ndarray) -> Prediction:
        mu, std = model.predict(X_candidates, return_std=True)
        return Prediction(mean=mu, std=std)